# Search the Alert Stream for Possible Unmatched SSOs

Author: James E. Robinson

This notebook gives a rough starting point for working with LSST alerts through the Fink broker.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import astropy.units as u
from astropy.time import Time
import io
import requests
from astropy.io import fits
from astropy.wcs import WCS
import glob
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_exact,reproject_interp,reproject_adaptive
from reproject.mosaicking import reproject_and_coadd
from astropy.visualization.wcsaxes import WCSAxes
from astropy import visualization as aviz

## Fink Data Transfer

Use the Fink Data Transfer service to query and filter alerts of interest: https://lsst.fink-portal.org/download

Documentation: https://doc.lsst.fink-broker.org/services/data_transfer/

Install and register with `fink-client`: https://github.com/astrolabsoftware/fink-client

Use the interface to build the query or load a previous query from a .yml. For this notebook we are interested in in exploring the alert stream for unidentified fast moving objects. The main components of this query are alerts with:
- not identified as a moving object (no `ssObjectId`)
- fast moving (`trailLength` > 2 arcsec)
- return only selected fields from the `DiaSource` table (minimise size of requested data)
The data transfer interface will set up a kafka stream and provide a `fink-client` command to run on your local machine to download the requested data.

In [ ]:
# read the data from the query as a pandas DataFrame
df_dia = pd.read_parquet("ftransfer_lsst_2026-08-06_682423")

In [ ]:
df_dia

In [ ]:
# LSST provides fluxes (not magnitudes) for diaSources because difference imaging can result in negative flux detections
# Calculate magnitude and uncertainties from the selected flux measurement (NB log10 warning)
# We have queried for trailFlux which is a psf fit convolved with linear motion; psfFlux, apFlux etc are also available
# See schema: https://sdm-schemas.lsst.io/apdb.html
# See Data Products Definition Document: https://lse-163.lsst.io/

col = "trail"
colFlux = col+"Flux"
colMag = col+"Mag"
colFluxErr = colFlux+"Err"
colMagErr = colMag+"Err"

df_dia[colMag] = (np.array(df_dia[colFlux])*u.nJy).to(u.ABmag).value
df_dia[colMagErr] = np.abs((2.5 / np.log(10)) * (np.array(df_dia[colFluxErr]) / np.array(df_dia[colFlux])))

In [ ]:
# check the time range covered by the alerts we have retrieved
print(Time(np.amin(df_dia["midpointMjdTai"]), format = "mjd", scale = "tai").utc.iso)
print(Time(np.amax(df_dia["midpointMjdTai"]), format = "mjd", scale = "tai").utc.iso)

In [ ]:
# NB the pipeline does not alert on sources moving > 10 deg/day; secret spy satellites :O
motion_limit = (10 * u.deg / u.day).to(u.arcsec/u.s).value * 30.

In [ ]:
# histogram of retrieved trailLength
x_plot = "trailLength"
df_plot = df_dia
print(len(df_plot))

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0])

ax1.hist(df_plot[x_plot], bins = 100, histtype = "step")

ax1.set_xlabel(x_plot)
ax1.set_ylabel("number")
ax1.set_yscale("log")

ax1.axvline(motion_limit, c = "r", label = "pipeline cut-off")

plt.show()

In [ ]:
# histogram of measured trailAngle
# compare abs(trailAngle) to the full range of -180 < trailAngle < 180 deg
x_plot = "trailAngle"
df_plot = df_dia
print(len(df_plot))

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0])

ax1.hist(np.abs(df_plot[x_plot]), bins = 100, histtype = "step")
ax1.set_xlabel("|{}|".format(x_plot))

# ax1.hist(df_plot[x_plot], bins = 100, histtype = "step")
# ax1.set_xlabel(x_plot)

ax1.set_ylabel("number")
# ax1.set_yscale("log")

plt.show()

In [ ]:
# scatter plot showing distribution of ra, dec of alerts across the sky
# NB this is just a quick cartesian plot, not a proper spherical sky projection
x_plot = "ra"
y_plot = "dec"
c_plot = "midpointMjdTai"
df_plot = df_dia
print(len(df_plot))

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0])

s1 = ax1.scatter(df_plot[x_plot], df_plot[y_plot],c = df_plot[c_plot])
cbar = plt.colorbar(s1)

ax1.set_xlabel(x_plot)
ax1.set_ylabel(y_plot)
cbar.set_label(c_plot)
ax1.set_aspect("equal")

plt.show()

In [ ]:
# run this cell to make plots interactive
%matplotlib widget

In [ ]:
# run this cell to make plots inline
%matplotlib inline

In [ ]:
# zoom around interactively and look for lines of detections?
# e.g. ra ~ 58.1, dec ~ -48.7

x_plot = "ra"
y_plot = "dec"
c_plot = "midpointMjdTai"
df_plot = df_dia.sort_values(c_plot)
print(len(df_plot))

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0])

s1 = ax1.scatter(df_plot[x_plot], df_plot[y_plot],c = df_plot[c_plot], s=10)
cbar = plt.colorbar(s1)

ax1.set_xlabel(x_plot)
ax1.set_ylabel(y_plot)
cbar.set_label(c_plot)
ax1.set_aspect("equal")

ax1.set_xlim(57.8,58.3)
ax1.set_ylim(-49.2,-48.2)

plt.show()

In [ ]:
# Isolate detections associated with the linear looking feature
# I used masks and histograms to narrow down detections based on ra, dec, trailAngle and visit

In [ ]:
ra_mask = (df_dia['ra']<58.3) & (df_dia['ra']>57.8)
dec_mask = (df_dia['dec']<-48.) & (df_dia['dec']>-49.2)
ang_mask = (df_dia['trailAngle'] > 163) & (df_dia['trailAngle'] < 168)
visit_mask = df_dia['visit'] == 2026022300129

dia_mask = (ra_mask & dec_mask) & ang_mask & visit_mask
df_dia[dia_mask]

In [ ]:
x_plot = "ra"
y_plot = "dec"
c_plot = "midpointMjdTai"

df_plot = df_dia[dia_mask].sort_values(c_plot)
print(len(df_plot))

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0])

s1 = ax1.scatter(df_plot[x_plot], df_plot[y_plot],c = df_plot[c_plot])
cbar = plt.colorbar(s1)

ax1.set_xlabel(x_plot)
ax1.set_ylabel(y_plot)
cbar.set_label(c_plot)

plt.show()


## Alert Cutout Images
Use the alert diaSourceIds to query Fink for more alert data, e.g. the cutout images

In [ ]:
dia = np.array(df_dia[dia_mask]['diaSourceId'])
dia

In [ ]:
save_dir = "cutouts_wcs"
if not os.path.isdir(save_dir):
    os.mkdir(save_dir)

In [ ]:
# Use the fink api to query fits cutout images for each alert: https://doc.lsst.fink-broker.org/services/api/imagesearch/
for i,diaId in enumerate(dia):

    diaId = str(diaId)
    fname_fits = '{}/{}.fits'.format(save_dir,diaId)
    print("query {} cutout".format(diaId))

    r = requests.post(
        "https://api.lsst.fink-portal.org/api/v1/cutouts",
        json={
            "diaSourceId": diaId,
            "kind": "Science",
            "output-format": "FITS",
        },
    )
    

    hdu = fits.open(io.BytesIO(r.content), ignore_missing_simple=True)
    # hdu.writeto(fname_fits,overwrite='True') # save fits file without correcting the wcs


    # add missing/updated WCS fields to the PRIMARY header, keeping all other fields as well
    data = hdu["PRIMARY"].data
    hdr = hdu["PRIMARY"].header
    wcs = WCS(hdr)
    wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"] # updating the wcs ctype will also fix some other keys
    hdr_wcs = wcs.to_header()

    # add new keys that are missing from the original header (or update ones that have changed)
    for x in list(hdr_wcs.keys()):
        if x not in hdr:
            # print("add {}".format(x))
            hdr[x] = (hdr_wcs[x],hdr_wcs.comments[x])
        else:
            if hdr_wcs[x] != hdr[x]:
                # print("update {}".format(x))
                hdr[x] = (hdr_wcs[x],hdr_wcs.comments[x])
    

    # update the hdu header and save to file
    hdu["PRIMARY"] = fits.PrimaryHDU(data,header=hdr)
    hdu.writeto(fname_fits,overwrite='True',output_verify='ignore') # NB need to ignore verification 

## Inspect the fits images

In [ ]:
wcs_files = glob.glob("{}/*.fits".format(save_dir))
wcs_files = np.sort(wcs_files)
wcs_files

In [ ]:
# inspect one of the cutouts
hdu = fits.open(wcs_files[0])
hdu.info()

In [ ]:
# inspect the header
hdu["PRIMARY"].header

In [ ]:
# check the WCS
WCS(hdu["PRIMARY"].header)

In [ ]:
# plot the fits image with WCS projection
# NB that North is probably not up as expected due to LSSTCam rotation
hdu = fits.open(wcs_files[0])["PRIMARY"]
plot_img = hdu.data
plot_wcs = WCS(hdu.header)

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0], projection = plot_wcs)

# plot image with normalisation
norm = aviz.ImageNormalize(plot_img,interval=aviz.ZScaleInterval())
s1 = ax1.imshow(plot_img, norm=norm)
plt.colorbar(s1)

ax1.grid(color='white', ls='solid')

plt.show()


In [ ]:
# reproject the fits image to get North up

wcs_out, shape_out = find_optimal_celestial_wcs([hdu], hdu_in = "PRIMARY")

array, footprint = reproject_exact(input_data = hdu,
                                       hdu_in = "PRIMARY",
                                       output_projection = wcs_out, shape_out=shape_out,
                                   )


In [ ]:
wcs_out, shape_out

In [ ]:
plot_img = array
plot_wcs = wcs_out

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0], projection = plot_wcs)

# plot image
norm = aviz.ImageNormalize(plot_img,interval=aviz.ZScaleInterval())
s1 = ax1.imshow(plot_img, norm=norm)
plt.colorbar(s1)

ax1.grid(color='white', ls='solid')

plt.show()


## Plot all cutouts together

In [ ]:
# reproject and mosaic all cutouts to get them on a common projection

hdus =  [fits.open(f) for f in wcs_files][:10]

wcs_out, shape_out = find_optimal_celestial_wcs(hdus, hdu_in = "PRIMARY")

array, footprint = reproject_and_coadd(input_data = hdus,
                                       hdu_in = "PRIMARY",
                                       output_projection = wcs_out, shape_out=shape_out,
                                       reproject_function=reproject_adaptive,
                                       # boundary_mode = 'constant',
                                        # boundary_fill_value=np.nan,
                                      )

# TODO: reproject_adpative should allow setting of non image pixels to nan (e.g. boundary_mode, boundary_fill_value) but I can't figure it out right now...
# Set non image pixels to nan so that only cutout pixels show up in mosaic
array[array == 0] = np.nan
array

In [ ]:
wcs_out,shape_out

In [ ]:
# get the centre coordinates of each cutout
ra = []
dec = []
for i in range(len(hdus)):
    ra.append(hdus[i]["PRIMARY"].header['CRVAL1'])
    dec.append(hdus[i]["PRIMARY"].header['CRVAL2'])
ra = np.array(ra)
dec = np.array(dec)


In [ ]:
%matplotlib widget

In [ ]:
%matplotlib inline

In [ ]:
# plot all cutouts combined into a single image
# zoom in and verify that the cutouts all have consistent trailAngle
# these detections are probably flashes from a rapidly rotating piece of space debris

plot_img = array
plot_wcs = wcs_out

fig = plt.figure()
gs = gridspec.GridSpec(1, 1)
ax1 = plt.subplot(gs[0,0], projection = plot_wcs)

# plot image
norm = aviz.ImageNormalize(plot_img,interval=aviz.ZScaleInterval())
s1 = ax1.imshow(plot_img, norm=norm)

# use marker to highlight cutout position
ax1.scatter(ra, dec, transform=ax1.get_transform('world'), 
            edgecolor = 'r', facecolor = 'none', s = 100)

# ax1.scatter(c_cent.ra,c_cent.dec, transform=ax1.get_transform('world'),
#           edgecolor = "r", facecolor = 'none', s =200)

plt.colorbar(s1)

plt.show()
